# Day C2 — convolution과 필터 설계: 커널을 밀어 보면 필터가 보인다

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/C-neural-signals/day-C2.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/C-neural-signals/day-C2.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 손 계산, `np.convolve`, FFT 곱셈은 같다

In [ ]:
import numpy as np

sig = np.array([0, 0, 1, 1, 1, 0, 0, 0], dtype=float)   # 짧은 상자 모양 신호
ker = np.array([1, 2, 1], dtype=float) / 4               # 작은 가중 평균 커널

# (1) 손으로: 커널을 뒤집어 한 칸씩 밀며 내적(dot product)
N, M = len(sig), len(ker)
padded = np.concatenate([np.zeros(M - 1), sig, np.zeros(M - 1)])
by_hand = np.array([padded[i:i + M] @ ker[::-1] for i in range(N + M - 1)])

# (2) numpy 함수
by_np = np.convolve(sig, ker)                            # 기본값 mode="full"

# (3) 주파수 영역: FFT끼리 곱한 뒤 역FFT (길이를 N+M-1로 맞춰야 한다)
L = N + M - 1
by_fft = np.real(np.fft.ifft(np.fft.fft(sig, L) * np.fft.fft(ker, L)))

print("결과 길이:", len(by_hand), "= N + M - 1 =", L)
print("손 계산 :", np.round(by_hand, 3))
print("np 함수 :", np.round(by_np, 3))
print("FFT 곱  :", np.round(by_fft, 3) + 0.0)
print("세 결과가 같은가:", np.allclose(by_hand, by_np) and np.allclose(by_np, by_fft))

# 날개(wing) 자르기: 앞뒤로 (M-1)/2 개씩 버리면 원래 길이 N, 시점도 맞는다
half = (M - 1) // 2
same = by_np[half:half + N]
print("날개 자른 결과:", np.round(same, 3), "| mode='same'과 같은가:", np.allclose(same, np.convolve(sig, ker, mode="same")))

### 예시 2 — 튀는 값이 있을 때: 이동 평균 vs 이동 중앙값

In [ ]:
import numpy as np
from scipy import signal

rng = np.random.default_rng(2)
fs = 200
t = np.arange(0, 3, 1 / fs)
clean = np.sin(2 * np.pi * 2 * t)                 # 느린 2 Hz 파형
x = clean + rng.normal(0, 0.3, len(t))
spike_idx = rng.choice(len(t), 8, replace=False)
x[spike_idx] += 6                                  # 드문 큰 튐(artifact) 8개

k = 11                                             # 창 길이 11개 = 55 ms (홀수로)
run_mean = np.convolve(x, np.ones(k) / k, mode="same")   # 이동 평균 = 상자 커널과의 convolution
run_med = signal.medfilt(x, kernel_size=k)               # 이동 중앙값

def err(y):
    inner = slice(k, -k)                           # 양 끝 날개 구간은 빼고 비교
    return np.sqrt(np.mean((y[inner] - clean[inner]) ** 2))

print("원 신호와의 RMS 오차: 원자료 %.3f | 이동 평균 %.3f | 이동 중앙값 %.3f" % (err(x), err(run_mean), err(run_med)))
i = spike_idx[0]
print("튐 위치 값  원자료 %.2f | 이동 평균 %.2f | 이동 중앙값 %.2f | 참값 %.2f" % (x[i], run_mean[i], run_med[i], clean[i]))

### 예시 3 — FIR와 IIR, 한 방향과 앞뒤

In [ ]:
import numpy as np
from scipy import signal

fs = 500
t = np.arange(-1, 2, 1 / fs)
# 0초에 시작해 0.4초 동안 이어지는 10 Hz 폭발(burst), 가우시안 포락선
env = np.exp(-((t - 0.2) ** 2) / (2 * 0.08 ** 2))
x = env * np.sin(2 * np.pi * 10 * t)

band = [8, 12]
# FIR: 계수 numtaps개짜리 커널. 길수록 주파수 경계가 날카롭다 (여기선 0.5초 = 251개)
fir = signal.firwin(251, band, pass_zero=False, fs=fs)
# IIR: Butterworth 4차, sos 형식이 수치적으로 안전하다
sos = signal.butter(4, band, btype="bandpass", fs=fs, output="sos")

def gain(f, which):
    if which == "fir":
        w, h = signal.freqz(fir, worN=[f], fs=fs)
    else:
        w, h = signal.sosfreqz(sos, worN=[f], fs=fs)
    return abs(h[0])

for f in [4, 10, 20]:
    print("%2d Hz 통과 비율  FIR %.3f | IIR %.3f" % (f, gain(f, "fir"), gain(f, "iir")))

def peak_time(y):
    return t[np.argmax(np.abs(signal.hilbert(y)))]   # 포락선이 가장 큰 시각

out = {
    "FIR 한 방향 (lfilter)": signal.lfilter(fir, 1, x),
    "FIR 앞뒤 (filtfilt)": signal.filtfilt(fir, 1, x),
    "IIR 한 방향 (sosfilt)": signal.sosfilt(sos, x),
    "IIR 앞뒤 (sosfiltfilt)": signal.sosfiltfilt(sos, x),
}
print("원 신호 포락선 최고점: %.3f초" % peak_time(x))
for name, y in out.items():
    print("%-22s 최고점 %.3f초 (지연 %+.0f ms)" % (name, peak_time(y), 1000 * (peak_time(y) - peak_time(x))))
print("FIR 이론 지연 = (numtaps-1)/2/fs = %.0f ms" % (1000 * (251 - 1) / 2 / fs))